# 서울 아파트 실거래가 수집기
**마포 / 용산 / 성동 / 광진 / 동대문 / 은평 / 서대문 / 양천 / 강서 / 영등포 / 동작 / 관악 / 강동 / 종로 · 2020~2026년**

순서대로 셀을 실행하면 됩니다. ▶ 버튼을 위에서부터 차례로 누르세요.

## 1단계 · 저장소 클론 및 패키지 설치

In [ ]:
# 저장소 클론 (이미 있으면 pull로 최신화)
import os
if os.path.isdir('realestate_analysis_tool'):
    %cd realestate_analysis_tool
    !git pull
else:
    !git clone https://github.com/jiuk96/realestate_analysis_tool.git
    %cd realestate_analysis_tool

!pip install -q -r requirements.txt
print('✅ 설치 완료')

## 2단계 · API 키 / GitHub 토큰 입력

> 아래 칸에 키를 입력하고 ▶ 를 누르세요. 키는 Colab 세션 메모리에만 저장되며 외부에 노출되지 않습니다.

In [ ]:
import os
from getpass import getpass

molit_key = getpass('국토교통부 API 키 입력: ')
git_token = getpass('GitHub Token 입력 (repo 권한): ')

with open('.env', 'w') as f:
    f.write(f'MOLIT_API_KEY={molit_key}\n')
    f.write(f'GIT_TOKEN={git_token}\n')

print('✅ 키 설정 완료 (.env 저장됨)')

## 3단계 · API 연결 테스트

In [ ]:
import os, requests, xml.etree.ElementTree as ET
from dotenv import load_dotenv
load_dotenv()

key = os.getenv('MOLIT_API_KEY')
url = (
    'http://apis.data.go.kr/1613000/RTMSDataSvcAptTrade/getRTMSDataSvcAptTrade'
    f'?serviceKey={key}&LAWD_CD=11440&DEAL_YMD=202001&pageNo=1&numOfRows=3'
)
resp = requests.get(url, timeout=15)
print(f'HTTP 상태코드: {resp.status_code}')
if resp.status_code == 200:
    root = ET.fromstring(resp.content)
    tc = root.find('.//totalCount')
    print(f'✅ API 연결 성공! 마포구 2020년 1월 거래 건수: {tc.text}건' if tc is not None else '✅ 연결 성공')
else:
    print(f'❌ 연결 실패: {resp.text[:300]}')

## 4단계 · 데이터 수집 실행

**수집 대상**: 14개 구 × 78개월(2020.01~2026.06) = 최대 1,092회 API 호출  
**예상 소요 시간**: 30~50분 (체크포인트 있으면 이미 수집된 달은 스킵)

| 구분 | 구 목록 |
|------|--------|
| 기존 (마용성) | 마포구, 용산구, 성동구 |
| 신규 추가 | 광진구, 동대문구, 은평구, 서대문구, 양천구, 강서구, 영등포구, 동작구, 관악구, 강동구, 종로구 |

In [ ]:
import sys
sys.path.insert(0, '.')

from src.collector import collect_all
df = collect_all()

if not df.empty:
    print(f'\n✅ 수집 완료!')
    print(f'   총 거래 건수 : {len(df):,}건')
    print(f'   단지 수      : {df["apt_name"].nunique()}개')
    print(f'   메모리 사용량: {df.memory_usage(deep=True).sum()/1024**2:.1f} MB')
    print()
    print('구별 거래 건수:')
    print(df.groupby('district_name', observed=True)['deal_amount'].count().sort_values(ascending=False).to_string())

## 5단계 · GitHub에 푸시

In [ ]:
from src.collector import git_push_data

!git config user.email 'jiuk9612@gmail.com'
!git config user.name 'jiuk96'

success = git_push_data('data: 서울 14개 구 실거래가 수집 완료 (2020~2026)')

if success:
    print('✅ GitHub 푸시 완료!')
    print('   → https://github.com/jiuk96/realestate_analysis_tool')
else:
    print('❌ 푸시 실패 — GitHub Token 권한(repo)을 확인해주세요.')

## (선택) 특정 구만 추가 수집

이미 마용성 데이터가 있고 신규 구만 추가하고 싶다면 아래 셀을 실행하세요.

In [ ]:
import sys
sys.path.insert(0, '.')
from src.collector import collect_all

# 신규 11개 구만 따로 수집
NEW_DISTRICTS = {
    '광진구':   '11215',
    '동대문구': '11230',
    '은평구':   '11380',
    '서대문구': '11410',
    '양천구':   '11470',
    '강서구':   '11500',
    '영등포구': '11560',
    '동작구':   '11590',
    '관악구':   '11620',
    '강동구':   '11740',
    '종로구':   '11110',
}

df_new = collect_all(districts=NEW_DISTRICTS)

if not df_new.empty:
    print(f'신규 수집: {len(df_new):,}건, {df_new["apt_name"].nunique()}개 단지')
    print(df_new.groupby('district_name', observed=True)['deal_amount'].count().sort_values(ascending=False).to_string())

## 수집 완료 후 다음 단계

Claude Code 환경에서 아래 명령어로 데이터를 받아 분석을 실행하세요:

```bash
git pull
python web/app.py   # → http://localhost:5000
```